## predicting-mortality-of-icu-patients-the-physionet-computing-in-cardiology-challenge-2012-1.0.0
### author--Yuyao Lei
### netid--yl1216

In [ ]:
#pip install tableone
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import missingno as msno
from tableone import TableOne

In [ ]:
#code: setup
ROOT=Path.cwd()
#an assertion that "checkpoint1.ipynb" is in it
assert(ROOT/"checkpoint1.ipynb").exists(),"checkpoint1.ipynb not found"
DATA=ROOT/"data"
OUT=ROOT/"output"
OUT.mkdir(exist_ok=True)

In [ ]:
#code: check data
def check_set_a(data_dir: Path):
    files=list((data_dir/"set-a").glob("*"))
    assert len(files)>0,"no file in set-a"
    return len(files)

n_files=check_set_a(DATA)
assert n_files==4000, f"expected 4000 records, but only {n_files} found"


In [ ]:
#code: load
def load_set_a(data_dir: Path):
    data_dir=Path(data_dir)
    all_long=[]
    #iterate through 4000 txt file records
    for file in data_dir.glob("*.txt"):
        #set the name of txt as sample_id
        sample_id=file.stem
        #read as csv
        df=pd.read_csv(file)
        df["id"]=sample_id
        all_long.append(df)
    long_df=pd.concat(all_long,ignore_index=True)
    wide_df=(long_df.pivot_table(index=["id","Time"],columns="Parameter",values="Value",aggfunc="mean").reset_index())
    return long_df, wide_df
long_df,wide_df=load_set_a(DATA/"set-a")
#check the structure of data frame
wide_df["id"] = wide_df["id"].astype(str)
long_df["id"] = long_df["id"].astype(str)
#make id str type, or error would occur when merging the data frame in table1
print(long_df.shape)
print(wide_df.shape)
print(long_df)
print(wide_df)
long_df.to_csv(OUT/ "set-a_long.csv",index=False)
wide_df.to_csv(OUT/ "set-a_wide.csv",index=False)


## what exists at prediction time
At prediction time, **early hospitalization information exists** and **future information (survival or death) is not yet observed**.

Information including:
- patient identifier (id)
- measured time (Time)
- basic demographic variables (e.g., age, gender)
- vital signs (e.g., HR, Temp)
- lab measurements (e.g., Urine, Creatinine)
- other recorded parameters

In long format, the following columns are included in the structure:
- id: patient identifier
- time: time since admission
- parameter: measured variable name
- value: measurement value

In wide formats, the first row includes every parameter name: 
- id
- Time
- other measurements
and the values of each variable are following after the column names.

**Only information available up to a fixed prediction time window (<47:36) is included.** Measurements occurring after this window are excluded to **avoid data leakage**.
Ensure that the prediction simulates a realistic clinical decision-making scenario.

In [ ]:
#code: load outcomes
#from pathlib import Path
#print(list(ROOT.rglob("Outcomes*")))
#print((DATA / "Outcomes-a.txt").exists())
outcomes_raw=pd.read_csv(DATA / "Outcomes-a.txt")
#Question when debuging: if there isn't a space in fornt of "Outcomes-a.txt", error occurs, but why?
outcomes_df=outcomes_raw[["RecordID","In-hospital_death"]].rename(columns={"RecordID":"id","In-hospital_death":"death"})
outcomes_df["id"] = outcomes_df["id"].astype(str)
#让id都是字符型数据
print(outcomes_df.head())
#outcome summary
print(outcomes_df["death"].value_counts())
print(outcomes_df["death"].mean())

## Outcome summary

The variables include:
- RecordID
- SAPS-I
- SOFA
- Length_of_stay
- Survival
- In-hospital_death

We only extract **RecordID** as **id** and **In-hospital_death** as **death**. The status of patients is recorded as **death=1, survival=0**. 

Print the **counts of deaths and survivals**, **the mean** of outcome summary shows the **proportion of deaths in addmission patients**.

The outcome label should not be available at prediction time.


In [ ]:
#Table1
wide_df_clean = wide_df.drop(columns=["Time"], errors="ignore")#missingmap in id order
wide_df_clean2=wide_df.drop(columns=["id"], errors="ignore")#missingmap in time order
#if column["Time"] not be dropped, warning occurs because the variable Time is not numeric.
table1_df = wide_df_clean.merge(outcomes_df, on="id", how="left")
columns = [c for c in table1_df.columns if c != "death"]
Table1 = TableOne(
    table1_df,
    columns=columns,
    categorical=[],   
    groupby="death",
    pval=True
)
Table1

## Table1: Grouped by in-hospital death
The table summarizes patient characteristics and lab measurements, grouped by in-hospital death.
### key observation
It reports:
- mean (SD) for each variable
- missing counts
- group-wise comparison
- *P*-value for statistical significance
### interpretation
- The distribution of several variables differs between the death and survival groups (e.g., AST, Creatinine), suggesting potential associations with patient outcomes.
- Variables with statistically significant *P*-values (p < 0.05) may indicate important predictors of in-hospital mortality, although these should be interpreted cautiously due to multiple comparisons and potential confounding.
- Some variables (e.g., MechVent) may show unstable or undefined *P*-values due to limited variation or small effective sample sizes.
- The magnitude of differences (in addition to *P*-values) should be considered when assessing clinical relevance.


In [ ]:
#missingness map
msno.matrix(wide_df_clean, figsize=(8, 3.0), fontsize=10, sparkline=False) 
plt.title("nullity matrix: rows in admission id order, white = missing") 
plt.tight_layout() 
plt.show()

## nullity matrix
This nullity matrix visualizes the presence (black) and absence (white) of measurements across variables for all observations.
Rows are id of patient in order, and each column represents a variable.
Overall, the nullity matrix reveals substantial and structured missingness across variables, indicating that missingness is likely informative rather than random.
### key observation
- Many lab variables exhibit substantial missingness.
- Some variables are consistently measured across observations (e.g., HR, Weight).
- The missingness pattern appears structured rather than random.
- Clusters of missingness across variables suggest that certain tests are ordered together, reflecting underlying clinical workflows.
### interpretation
- measurements are not consistently collected across patients, given that missingness is common in some variables.
- consistently measured variables suggest that they are part of routine monitoring.
- The observed missingness patterns suggest that the data are unlikely to be Missing Completely At Random (MCAR). Instead, missingness may depend on patient condition or clinical decisions, which has important implications for downstream modeling.

In [ ]:
#missingness informative
count_df = (
    long_df
    .groupby(["id", "Parameter"])
    .size()
    .reset_index(name="n_measurements")
)
count_wide = count_df.pivot(
    index="id",
    columns="Parameter",
    values="n_measurements"
).fillna(0)
long_df["id"] = long_df["id"].astype(str)
outcomes_df["id"] = outcomes_df["id"].astype(str)
#unify the type of id in different data frames, or error occurs when merge
plot_df = count_wide.merge(outcomes_df, on="id", how="left")
parameters = count_wide.columns

for p in parameters:
    temp = plot_df[["death", p]].copy()
    temp.columns = ["death", "count"]

    plt.figure(figsize=(5, 3))
    sns.boxplot(data=temp, x="death", y="count")

    plt.title(f"{p}: informative presence")
    plt.xlabel("death (0=survived, 1=died)")
    plt.ylabel("number of measurements")

    plt.tight_layout()
    plt.show()

## missingness informative
The number of times each lab was measured predicts in-hospital death.
Before looking at a single lab value: clinicians order a test when they are worried, and their worry is informative.
This is why “focus on what is observed, not what is missing”. The Informed Presence is also a leakage trap: the lab measurement was ordered because someone already suspected the diagnosis.
### key observation
- The boxplot shows the distribution of the number of each measurements per patient, stratified by in-hospital death (0 = survived, 1 = died).
- The y-axis represents the number of times each measurement was measured for each patient.
- For some measurements (e.g., ALT, ALP), patients who died tend to have more frequent measurements compared to those who survived.
- For some measurements (e.g., AST, Albumin), a large proportion of patients have zero measurements, indicating that this test is not routinely performed for all patients.
- For some measurements (e.g., SysABP, WBC), the frequency of measurement did not differ from between survived and died group; the presence of some measurements is not such informative (e.g., FiO2). 
- The death group often shows more extreme values (e.g., cholesterol, Creatinine), suggesting that some patients underwent repeated testing, possibly due to clinical deterioration.
### interpretation
- Measurement frequency is likely driven by clinical decisions, where sicker patients are monitored more frequently.
- For informative presence, missingness is unlikely to be random and may reflect underlying patient severity (i.e., Missing Not At Random, MNAR).
- The number of measurements itself may carry predictive information, independent of the measured values.
- If the boxplot indicates that measurement frequency differs by outcome, suggesting that missingness is informative and related to patient severity.

## AI-use statement
AI tools (e.g., ChatGPT) was used to assist with:
- Understand assignment requirements and provide an outline of the approach for the assignment
- Interpret, structure, and debug Python code
- Provide guidance on the installation and setup of Pixi, Git, and other terminal tools
All outputs generated by AI were carefully reviewed, modified, verified, and adapted to fit the context of this assignment. I ensured that I understand the code and analysis presented, and I take full responsibility for all submitted work.